In [0]:
%sql
-- Compare key metrics between the view and the prod table for a single day
SELECT
  source,
  COUNT(*)                             AS total_rows,
  COUNT(DISTINCT tvid)                 AS distinct_tvids,
  COUNT(DISTINCT (tvid, ts_start, ts_end)) AS distinct_sessions,
  COUNT(DISTINCT value)                AS distinct_commercials
FROM (
  -- View output
  SELECT 'view: test_enhanced_kinetiq_refined' AS source, tvid, ts_start, ts_end, value
  FROM dev.mohit_gangwani.test_enhanced_kinetiq_refined
  WHERE date(ts_start) = '2026-07-20'

  UNION ALL

  -- Prod table
  SELECT 'prod: enhanced_kinetiq_commercial' AS source, tvid, ts_start, ts_end, value
  FROM prod.datalake_share.enhanced_kinetiq_commercial
  WHERE date_partition = '2026-07-20'
)
GROUP BY source
ORDER BY source


In [0]:
%sql
-- Isolate how much of the view's data comes from detection_onn (extra source not in prod table)
-- The prod table only reads from Kinetiq's S3 CSV feed; the view adds detection_onn on top of detection
SELECT
  source,
  COUNT(*)                             AS total_rows,
  COUNT(DISTINCT tvid)                 AS distinct_tvids,
  COUNT(DISTINCT (tvid, session_start, session_end)) AS distinct_sessions,
  COUNT(DISTINCT external_id)          AS distinct_commercials
FROM (
  -- detection only (mirror of what the view reads as first leg)
  SELECT 'detection (non-ONN)' AS source, tvid, session_start, session_end, external_id
  FROM prod.detection.viewing_commercials_dedup_golden
  WHERE date(session_start) = '2026-07-20'
    AND NOT ((COALESCE(acrb_clients, '||') != '||' AND acrb_clients NOT LIKE '%|vizio|%')
             OR acrb_clients <=> '|ALL|')
    AND fk_commercial_source_id = 1
    AND commercial_client = 'kinetiq'

  UNION ALL

  -- detection_onn only (extra source not present in prod table)
  SELECT 'detection_onn (extra)' AS source, tvid, session_start, session_end, external_id
  FROM prod.detection_onn.viewing_commercials_dedup_golden
  WHERE date(session_start) = '2026-07-20'
    AND NOT ((COALESCE(acrb_clients, '||') != '||' AND acrb_clients NOT LIKE '%|vizio|%')
             OR acrb_clients <=> '|ALL|')
    AND fk_commercial_source_id = 1
    AND commercial_client = 'kinetiq'
)
GROUP BY source
ORDER BY source


In [0]:
%sql
-- Which TVIDs appear in the view but are absent from the prod table for the same day?
-- Tags each "view-only" TVID with which source leg introduced it (detection vs detection_onn)
SELECT
  source_leg,
  COUNT(DISTINCT tvid) AS tvids_only_in_view,
  COUNT(*)             AS rows_only_in_view
FROM (
  SELECT source_leg, tvid
  FROM (
    SELECT 'detection (non-ONN)' AS source_leg, tvid
    FROM prod.detection.viewing_commercials_dedup_golden
    WHERE date(session_start) = '2026-07-20'
      AND NOT ((COALESCE(acrb_clients, '||') != '||' AND acrb_clients NOT LIKE '%|vizio|%')
               OR acrb_clients <=> '|ALL|')
      AND fk_commercial_source_id = 1
      AND commercial_client = 'kinetiq'

    UNION ALL

    SELECT 'detection_onn (extra)' AS source_leg, tvid
    FROM prod.detection_onn.viewing_commercials_dedup_golden
    WHERE date(session_start) = '2026-07-20'
      AND NOT ((COALESCE(acrb_clients, '||') != '||' AND acrb_clients NOT LIKE '%|vizio|%')
               OR acrb_clients <=> '|ALL|')
      AND fk_commercial_source_id = 1
      AND commercial_client = 'kinetiq'
  ) view_data
  WHERE tvid NOT IN (
    SELECT DISTINCT tvid
    FROM prod.datalake_share.enhanced_kinetiq_commercial
    WHERE date_partition = '2026-07-22'
  )
)
GROUP BY source_leg
ORDER BY source_leg


In [0]:
%sql
-- The prod table resolves TVIDs via: split(map.tvid, '_')[0]
-- (stripping suffixes like _2, _3 from hash-mapped TVIDs)
-- Check if detection-table TVIDs contain suffixes that inflate the view's TVID count
SELECT
  CASE WHEN tvid LIKE '%\_%' THEN 'has_suffix (tvid_xxx_N)'
       ELSE 'no_suffix'
  END                    AS tvid_format,
  COUNT(DISTINCT tvid)   AS distinct_raw_tvids,
  COUNT(DISTINCT split(tvid, '_')[0]) AS distinct_stripped_tvids,
  COUNT(*)               AS total_rows
FROM prod.detection.viewing_commercials_dedup_golden
WHERE date(session_start) = '2026-07-20'
  AND NOT ((COALESCE(acrb_clients, '||') != '||' AND acrb_clients NOT LIKE '%|vizio|%')
           OR acrb_clients <=> '|ALL|')
  AND fk_commercial_source_id = 1
  AND commercial_client = 'kinetiq'
GROUP BY 1
ORDER BY 1


In [0]:
%sql
-- Check whether any TVID in the prod table for 2026-07-20 also appears in detection_onn
-- If the prod table is truly ONN-free, the onn_tvids_in_prod count should be 0
SELECT
  'prod TVIDs also in detection_onn'   AS check_label,
  COUNT(DISTINCT p.tvid)               AS tvid_count,
  COUNT(*)                             AS row_count
FROM prod.datalake_share.enhanced_kinetiq_commercial p
INNER JOIN (
  SELECT DISTINCT tvid
  FROM prod.detection_onn.viewing_commercials_dedup_golden
  WHERE date(session_start) = '2026-07-20'
    AND NOT ((COALESCE(acrb_clients, '||') != '||' AND acrb_clients NOT LIKE '%|vizio|%')
             OR acrb_clients <=> '|ALL|')
    AND fk_commercial_source_id = 1
    AND commercial_client = 'kinetiq'
) onn ON p.tvid = onn.tvid
WHERE p.date_partition = '2026-07-20'

UNION ALL

SELECT
  'prod TVIDs NOT in detection_onn'    AS check_label,
  COUNT(DISTINCT p.tvid)               AS tvid_count,
  COUNT(*)                             AS row_count
FROM prod.datalake_share.enhanced_kinetiq_commercial p
LEFT ANTI JOIN (
  SELECT DISTINCT tvid
  FROM prod.detection_onn.viewing_commercials_dedup_golden
  WHERE date(session_start) = '2026-07-20'
    AND NOT ((COALESCE(acrb_clients, '||') != '||' AND acrb_clients NOT LIKE '%|vizio|%')
             OR acrb_clients <=> '|ALL|')
    AND fk_commercial_source_id = 1
    AND commercial_client = 'kinetiq'
) onn ON p.tvid = onn.tvid
WHERE p.date_partition = '2026-07-20'


In [0]:
%sql
-- Check whether any TVID in the prod table for 2026-07-20 also appears in detection_onn
-- If the prod table is truly ONN-free, the onn_tvids_in_prod count should be 0
SELECT
  'prod TVIDs also in detection_onn'   AS check_label,
  COUNT(DISTINCT p.tvid)               AS onn_tvids_in_prod,
  COUNT(*)                             AS onn_rows_in_prod
FROM prod.datalake_share.enhanced_kinetiq_commercial p
INNER JOIN (
  SELECT DISTINCT tvid
  FROM prod.detection_onn.viewing_commercials_dedup_golden
  WHERE date(session_start) = '2026-07-20'
    AND NOT ((COALESCE(acrb_clients, '||') != '||' AND acrb_clients NOT LIKE '%|vizio|%')
             OR acrb_clients <=> '|ALL|')
    AND fk_commercial_source_id = 1
    AND commercial_client = 'kinetiq'
) onn ON p.tvid = onn.tvid
WHERE p.date_partition = '2026-07-20'

UNION ALL

SELECT
  'prod TVIDs NOT in detection_onn (expected non-ONN)' AS check_label,
  COUNT(DISTINCT p.tvid)               AS onn_tvids_in_prod,
  COUNT(*)                             AS onn_rows_in_prod
FROM prod.datalake_share.enhanced_kinetiq_commercial p
LEFT ANTI JOIN (
  SELECT DISTINCT tvid
  FROM prod.detection_onn.viewing_commercials_dedup_golden
  WHERE date(session_start) = '2026-07-20'
    AND NOT ((COALESCE(acrb_clients, '||') != '||' AND acrb_clients NOT LIKE '%|vizio|%')
             OR acrb_clients <=> '|ALL|')
    AND fk_commercial_source_id = 1
    AND commercial_client = 'kinetiq'
) onn ON p.tvid = onn.tvid
WHERE p.date_partition = '2026-07-20'
